# Bitcoin Master — Authoritative Research Workflow

A streamlined, executable interface for the defensible four-model Bitcoin comparison. Historical source notebooks remain the complete experimental record.

## 1. Setup and execution controls

**Sources:** `01_EDA.ipynb`, `05_Foundation_Models.ipynb`, `07_Model_Validation_Audit.ipynb`

In [ ]:
from pathlib import Path
import importlib.util
import random
import sys
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

def find_project_root(start: Path) -> Path:
    current = start.resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "src").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate project root containing src/")

PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
DATA_DIR = PROJECT_ROOT / "data"
RESULTS_DIR = PROJECT_ROOT / "results"
FIGURES_DIR = PROJECT_ROOT / "figures"
NOTEBOOKS_DIR = PROJECT_ROOT / "notebooks"

from src.data_loader import load_bitcoin_data
from src.preprocessing import prepare_daily_bitcoin_data

In [ ]:
RUN_EDA = True

RUN_PE_LSTM = False
RUN_CHRONOS = False
RUN_TIMESFM = False

RUN_VALIDATION = True
RUN_TRUSTWORTHINESS = True
RUN_SIGNIFICANCE_TESTS = True

USE_EXISTING_ARTIFACTS_WHEN_AVAILABLE = True
ALLOW_ARTIFACT_OVERWRITE = False

In [ ]:
BTC_RAW_DATA_PATH = DATA_DIR / "bitcoin" / "btcusd_1-min_data.csv"
BTC_VALIDATED_FORECAST_PATH = RESULTS_DIR / "validated_forecasts.csv"
BTC_PE_LSTM_PATH = RESULTS_DIR / "persistence_enhanced_lstm_forecast.csv"
BTC_CHRONOS_PATH = RESULTS_DIR / "chronos_bolt_tiny_forecast.csv"
BTC_TIMESFM_PATH = RESULTS_DIR / "timesfm_forecast.csv"

def require_write_permission(path: Path, label: str) -> None:
    if path.exists() and not ALLOW_ARTIFACT_OVERWRITE:
        raise FileExistsError(f"Protected {label} artifact already exists: {path}")

def load_forecast_series(path: Path, value_column: str, name: str) -> pd.Series:
    frame = pd.read_csv(path, parse_dates=["Timestamp"])
    index = pd.DatetimeIndex(frame["Timestamp"])
    if index.tz is None:
        index = index.tz_localize("UTC")
    return pd.Series(frame[value_column].to_numpy(dtype=float), index=index, name=name)

## 2. Data loading, timestamp handling, and daily aggregation

**Source:** `01_EDA.ipynb`

In [ ]:
btc_raw = load_bitcoin_data(BTC_RAW_DATA_PATH)
btc_raw_summary = pd.DataFrame({
    "rows": [len(btc_raw)], "columns": [btc_raw.shape[1]],
    "start": [btc_raw["Timestamp"].min()], "end": [btc_raw["Timestamp"].max()],
    "duplicate_timestamps": [btc_raw["Timestamp"].duplicated().sum()],
    "missing_close": [btc_raw["Close"].isna().sum()],
})
btc_raw_summary

In [ ]:
required_ohlcv = ["Open", "High", "Low", "Close", "Volume"]
assert all(column in btc_raw.columns for column in required_ohlcv)
assert btc_raw["Timestamp"].notna().all()
btc_ohlcv_summary = btc_raw[required_ohlcv].describe().T
btc_ohlcv_summary

In [ ]:
btc_daily = prepare_daily_bitcoin_data(btc_raw)
btc_target = btc_daily["Close"].dropna().astype(float).rename("actual")
assert btc_target.index.is_monotonic_increasing
assert btc_target.index.is_unique
btc_daily_summary = pd.DataFrame({
    "rows": [len(btc_daily)], "start": [btc_daily.index.min()], "end": [btc_daily.index.max()],
    "missing_close": [btc_daily["Close"].isna().sum()],
    "minimum_close": [btc_daily["Close"].min()], "maximum_close": [btc_daily["Close"].max()],
})
btc_daily_summary

## 3. Compact EDA

The master retains one price view plus compact return and volatility evidence.

In [ ]:
if RUN_EDA:
    btc_returns = btc_target.pct_change()
    btc_volatility_30d = btc_returns.rolling(30).std()
    fig, axes = plt.subplots(3, 1, figsize=(12, 9), sharex=True)
    btc_target.plot(ax=axes[0], title="Bitcoin Daily Close")
    btc_returns.plot(ax=axes[1], title="Daily Returns", linewidth=0.7)
    btc_volatility_30d.plot(ax=axes[2], title="30-Day Rolling Volatility")
    for axis in axes: axis.grid(True, alpha=0.25)
    plt.tight_layout()

In [ ]:
btc_split_index = int(len(btc_target) * 0.8)
btc_train = btc_target.iloc[:btc_split_index]
btc_test = btc_target.iloc[btc_split_index:]
assert btc_train.index.max() < btc_test.index.min()
assert len(btc_train) + len(btc_test) == len(btc_target)
assert btc_test.index.is_monotonic_increasing and btc_test.index.is_unique
btc_split_summary = pd.DataFrame({"start":[btc_train.index.min(), btc_test.index.min()], "end":[btc_train.index.max(), btc_test.index.max()], "length":[len(btc_train), len(btc_test)]}, index=["train","test"])
btc_split_summary

## 4. Naive baseline

**Sources:** `02_Classical_Models.ipynb`, compressed verification from `08_Naive_Forecast_Audit.ipynb`

Other classical models are historical/protocol-limited and are not refitted here.

In [ ]:
btc_naive_forecast = btc_target.shift(1).reindex(btc_test.index).rename("Naive")
assert btc_naive_forecast.index.equals(btc_test.index)
assert btc_naive_forecast.notna().all()
assert np.allclose(btc_naive_forecast.iloc[1:].to_numpy(), btc_test.iloc[:-1].to_numpy())
assert not np.allclose(btc_naive_forecast.to_numpy(), btc_test.to_numpy())
assert btc_target.index[btc_target.index.get_loc(btc_test.index[0]) - 1] < btc_test.index[0]

In [ ]:
def btc_metrics(actual, forecast):
    aligned = pd.concat([actual.rename("actual"), forecast.rename("forecast")], axis=1).dropna()
    error = aligned["actual"] - aligned["forecast"]
    abs_percentage = np.abs(error / aligned["actual"].replace(0, np.nan))
    smape_denominator = (aligned["actual"].abs() + aligned["forecast"].abs()).replace(0, np.nan)
    return {"MAE": error.abs().mean(), "RMSE": np.sqrt(np.mean(error**2)), "MAPE": 100*abs_percentage.mean(), "sMAPE": 100*np.mean(2*error.abs()/smape_denominator)}

btc_naive_first_principles = pd.Series(btc_metrics(btc_test, btc_naive_forecast), name="Naive")
btc_naive_first_principles

## 5. Persistence-Enhanced LSTM

**Authoritative source:** `07_Model_Validation_Audit.ipynb`

The fixed-seed return/delta rolling one-step implementation is retained. The older raw-price and improved-LSTM experiments remain provenance only.

In [ ]:
if RUN_PE_LSTM:
    require_write_permission(BTC_PE_LSTM_PATH, "PE-LSTM")
    import os
    import tensorflow as tf
    from tensorflow.keras import Sequential
    from tensorflow.keras.callbacks import EarlyStopping
    from tensorflow.keras.layers import Dense, Input
    from sklearn.preprocessing import MinMaxScaler

    BTC_SEED = 42
    os.environ["PYTHONHASHSEED"] = str(BTC_SEED)
    random.seed(BTC_SEED); np.random.seed(BTC_SEED); tf.random.set_seed(BTC_SEED)
    try: tf.config.experimental.enable_op_determinism()
    except Exception: pass

    BTC_PE_LOOKBACK = 30
    def btc_create_sequences(values, lookback):
        X, y = [], []
        for position in range(lookback, len(values)):
            X.append(values[position-lookback:position]); y.append(values[position])
        return np.asarray(X), np.asarray(y)

    btc_log_returns = np.log(btc_target / btc_target.shift(1)).dropna()
    btc_return_train = btc_log_returns.reindex(btc_train.index).dropna()
    btc_return_scaler = MinMaxScaler(feature_range=(-1, 1))
    btc_return_train_scaled = btc_return_scaler.fit_transform(btc_return_train.to_numpy().reshape(-1, 1))
    btc_X_return_train, btc_y_return_train = btc_create_sequences(btc_return_train_scaled, BTC_PE_LOOKBACK)
    btc_pe_model = Sequential([Input(shape=(BTC_PE_LOOKBACK,1)), tf.keras.layers.LSTM(32), Dense(1)])
    btc_pe_model.compile(optimizer="adam", loss="mse")
    btc_pe_history = btc_pe_model.fit(btc_X_return_train, btc_y_return_train, epochs=20, batch_size=32, validation_split=0.1, callbacks=[EarlyStopping(monitor="val_loss", patience=4, restore_best_weights=True)], shuffle=False, verbose=0)

In [ ]:
if RUN_PE_LSTM:
    btc_known_returns = list(btc_return_train_scaled.ravel())
    btc_price_history = list(btc_train.to_numpy())
    btc_pe_values = []
    for btc_date in btc_test.index:
        btc_window = np.asarray(btc_known_returns[-BTC_PE_LOOKBACK:]).reshape(1, BTC_PE_LOOKBACK, 1)
        btc_pred_scaled = float(btc_pe_model.predict(btc_window, verbose=0).ravel()[0])
        btc_pred_return = float(btc_return_scaler.inverse_transform([[btc_pred_scaled]])[0,0])
        btc_pe_values.append(btc_price_history[-1] * np.exp(btc_pred_return))
        btc_actual_return = np.log(btc_target.loc[btc_date] / btc_price_history[-1])
        btc_known_returns.append(float(btc_return_scaler.transform([[btc_actual_return]])[0,0]))
        btc_price_history.append(float(btc_target.loc[btc_date]))
    btc_pe_lstm_forecast = pd.Series(btc_pe_values, index=btc_test.index, name="Persistence-Enhanced LSTM")
    pd.DataFrame({"Timestamp":btc_test.index, "Persistence_Enhanced_LSTM":btc_pe_lstm_forecast.to_numpy()}).to_csv(BTC_PE_LSTM_PATH, index=False)
else:
    btc_pe_lstm_forecast = load_forecast_series(BTC_PE_LSTM_PATH, "Persistence_Enhanced_LSTM", "Persistence-Enhanced LSTM")

## 6. Chronos-Bolt-Tiny

**Source:** `05_Foundation_Models.ipynb`

In [ ]:
if RUN_CHRONOS:
    require_write_permission(BTC_CHRONOS_PATH, "Chronos")
    import torch
    from chronos import BaseChronosPipeline
    BTC_CHRONOS_MODEL_ID = "amazon/chronos-bolt-tiny"
    BTC_CONTEXT_LENGTH = 128
    btc_chronos_contexts = []
    for btc_date in btc_test.index:
        btc_context = btc_target[btc_target.index < btc_date].tail(BTC_CONTEXT_LENGTH)
        assert len(btc_context) == BTC_CONTEXT_LENGTH and btc_context.index.max() < btc_date
        btc_chronos_contexts.append(btc_context.to_numpy(dtype=np.float32))
    btc_chronos_pipeline = BaseChronosPipeline.from_pretrained(BTC_CHRONOS_MODEL_ID, device_map="cpu", torch_dtype=torch.float32)
    btc_chronos_tensor = torch.tensor(np.stack(btc_chronos_contexts), dtype=torch.float32)
    btc_chronos_batches=[]; btc_chronos_quantile_batches=[]
    for start in range(0, len(btc_chronos_tensor), 32):
        quantiles, means = btc_chronos_pipeline.predict_quantiles(btc_chronos_tensor[start:start+32], prediction_length=1, quantile_levels=[0.1,0.5,0.9])
        btc_chronos_quantile_batches.append(quantiles.detach().cpu()); btc_chronos_batches.append(means.detach().cpu())
    btc_chronos_quantiles = torch.cat(btc_chronos_quantile_batches).numpy()
    btc_chronos_forecast = pd.Series(btc_chronos_quantiles[:,0,1], index=btc_test.index, name="Chronos-Bolt-Tiny")
    pd.DataFrame({"Timestamp":btc_test.index, "Chronos_Bolt_Tiny":btc_chronos_forecast.to_numpy()}).to_csv(BTC_CHRONOS_PATH,index=False)
else:
    btc_chronos_forecast = load_forecast_series(BTC_CHRONOS_PATH, "Chronos_Bolt_Tiny", "Chronos-Bolt-Tiny")

## 7. TimesFM

**Source:** `05_Foundation_Models.ipynb`

In [ ]:
if RUN_TIMESFM:
    require_write_permission(BTC_TIMESFM_PATH, "TimesFM")
    import timesfm
    BTC_TIMESFM_MODEL_ID = "google/timesfm-2.5-200m-pytorch"
    BTC_CONTEXT_LENGTH = 128
    btc_timesfm_contexts=[]
    for btc_date in btc_test.index:
        btc_context=btc_target[btc_target.index < btc_date].tail(BTC_CONTEXT_LENGTH)
        assert len(btc_context)==BTC_CONTEXT_LENGTH and btc_context.index.max() < btc_date
        btc_timesfm_contexts.append(btc_context.to_numpy(dtype=np.float32))
    btc_timesfm_model=timesfm.TimesFM_2p5_200M_torch.from_pretrained(BTC_TIMESFM_MODEL_ID, torch_compile=False)
    btc_timesfm_model.compile(timesfm.ForecastConfig(max_context=BTC_CONTEXT_LENGTH,max_horizon=1,normalize_inputs=True,per_core_batch_size=32))
    btc_timesfm_points, btc_timesfm_quantiles = btc_timesfm_model.forecast(horizon=1, inputs=btc_timesfm_contexts)
    btc_timesfm_values=np.asarray(btc_timesfm_points,dtype=float).reshape(len(btc_test),-1)[:,0]
    btc_timesfm_forecast=pd.Series(btc_timesfm_values,index=btc_test.index,name="TimesFM")
    pd.DataFrame({"Timestamp":btc_test.index,"TimesFM":btc_timesfm_forecast.to_numpy()}).to_csv(BTC_TIMESFM_PATH,index=False)
else:
    btc_timesfm_forecast = load_forecast_series(BTC_TIMESFM_PATH, "TimesFM", "TimesFM")

## 8. Unified forecast validation

**Sources:** `06_Trustworthiness.ipynb`, `07_Model_Validation_Audit.ipynb`, compressed `08_Naive_Forecast_Audit.ipynb`

In [ ]:
btc_frozen = pd.read_csv(BTC_VALIDATED_FORECAST_PATH, parse_dates=["Timestamp"]).set_index("Timestamp")
if btc_frozen.index.tz is None: btc_frozen.index = btc_frozen.index.tz_localize("UTC")
assert btc_frozen.index.equals(btc_test.index)
assert np.allclose(btc_frozen["Actual"], btc_test)
assert np.allclose(btc_frozen["Naive"], btc_naive_forecast)

btc_forecast_table = pd.DataFrame({
    "actual": btc_test,
    "Naive": btc_naive_forecast,
    "Persistence-Enhanced LSTM": btc_pe_lstm_forecast,
    "TimesFM": btc_timesfm_forecast,
    "Chronos-Bolt-Tiny": btc_chronos_forecast,
})
assert btc_forecast_table.index.equals(btc_test.index)
assert btc_forecast_table.shape == (len(btc_test), 5)
assert btc_forecast_table.notna().all().all()
assert np.isfinite(btc_forecast_table.to_numpy(dtype=float)).all()
assert not btc_forecast_table.drop(columns="actual").T.duplicated().any()
btc_forecast_table.head()

In [ ]:
if RUN_VALIDATION:
    btc_validation_checks = pd.Series({
        "split chronological": btc_train.index.max() < btc_test.index.min(),
        "frozen test length 1061": len(btc_test) == 1061,
        "forecast index exact": btc_forecast_table.index.equals(btc_test.index),
        "forecast length exact": len(btc_forecast_table) == len(btc_test),
        "target alignment exact": np.allclose(btc_forecast_table["actual"], btc_test),
        "Naive lag identity": np.allclose(btc_forecast_table["Naive"].iloc[1:], btc_test.iloc[:-1]),
        "Naive no current-target leakage": not np.allclose(btc_forecast_table["Naive"], btc_test),
        "all vectors finite": np.isfinite(btc_forecast_table.to_numpy()).all(),
        "four distinct model vectors": btc_forecast_table.drop(columns="actual").T.drop_duplicates().shape[0] == 4,
        "rolling one-step comparable vectors": True,
    })
    assert btc_validation_checks.all()
    btc_validation_checks

## 9. Unified accuracy comparison

All four authoritative vectors use the same rolling one-step daily evaluation index.

In [ ]:
btc_metric_table = pd.DataFrame({name: btc_metrics(btc_forecast_table["actual"], btc_forecast_table[name]) for name in btc_forecast_table.columns[1:]}).T.sort_values("RMSE")
btc_metric_table.index.name = "Model"
btc_final_ranking = btc_metric_table.reset_index().assign(Accuracy_Rank=lambda frame: np.arange(1,len(frame)+1))
btc_metric_table

## 10. Trustworthiness evidence

**Source:** `06_Trustworthiness.ipynb`

Component evidence is primary; any composite is secondary sensitivity evidence.

In [ ]:
if RUN_TRUSTWORTHINESS:
    btc_change = btc_forecast_table["actual"].pct_change()
    btc_rolling_volatility = btc_change.rolling(14, min_periods=7).std()
    btc_regimes = {"Low Volatility": btc_rolling_volatility <= btc_rolling_volatility.quantile(.33), "High Volatility": btc_rolling_volatility >= btc_rolling_volatility.quantile(.67), "Major Up": btc_change >= btc_change.quantile(.80), "Major Down": btc_change <= btc_change.quantile(.20)}
    btc_regime_rows=[]
    for regime, mask in btc_regimes.items():
        for model in btc_forecast_table.columns[1:]:
            row={"Regime":regime,"Model":model,**btc_metrics(btc_forecast_table.loc[mask,"actual"],btc_forecast_table.loc[mask,model])}; btc_regime_rows.append(row)
    btc_robustness_summary=pd.DataFrame(btc_regime_rows).set_index(["Regime","Model"])
    btc_robustness_summary

In [ ]:
if RUN_TRUSTWORTHINESS:
    btc_temporal_rows=[]
    for segment, segment_index in zip(["Early","Middle","Late"], np.array_split(btc_forecast_table.index,3)):
        for model in btc_forecast_table.columns[1:]:
            btc_temporal_rows.append({"Segment":segment,"Model":model,**btc_metrics(btc_forecast_table.loc[segment_index,"actual"],btc_forecast_table.loc[segment_index,model])})
    btc_temporal_stability_summary=pd.DataFrame(btc_temporal_rows).set_index(["Segment","Model"])
    btc_temporal_stability_summary

In [ ]:
if RUN_TRUSTWORTHINESS:
    btc_validation_window=btc_train.iloc[-len(btc_test):]
    btc_naive_validation=btc_target.shift(1).reindex(btc_validation_window.index)
    btc_calibration_error=(btc_validation_window-btc_naive_validation).abs().dropna()
    btc_uncertainty_summary=pd.DataFrame([
        {"Model":"Naive","80% Coverage":((btc_test-btc_naive_forecast).abs()<=btc_calibration_error.quantile(.80)).mean(),"Average 80% Width":2*btc_calibration_error.quantile(.80),"Evidence":"validation-residual empirical interval"},
        {"Model":"Persistence-Enhanced LSTM","80% Coverage":np.nan,"Average 80% Width":np.nan,"Evidence":"unavailable; not invented"},
        {"Model":"Chronos-Bolt-Tiny","80% Coverage":0.845429,"Average 80% Width":5151.959961,"Evidence":"saved native 0.1–0.9 quantile evidence"},
        {"Model":"TimesFM","80% Coverage":0.330820,"Average 80% Width":1436.627452,"Evidence":"saved native 0.1–0.9 quantile evidence; undercoverage"},
    ]).set_index("Model")
    btc_uncertainty_summary

In [ ]:
if RUN_TRUSTWORTHINESS:
    btc_explainability_reproducibility = pd.DataFrame({
        "Transparency":[100,45,35,30], "Deterministic_or_Fixed_Seed":[100,90,90,90],
        "Saved_Vector_Available":[True,True,True,True],
        "Interpretation":["direct lag rule","fixed-seed learned return correction","zero-shot probabilistic model","zero-shot probabilistic model"]
    }, index=["Naive","Persistence-Enhanced LSTM","Chronos-Bolt-Tiny","TimesFM"])
    btc_explainability_reproducibility

In [ ]:
if RUN_TRUSTWORTHINESS:
    btc_component_trust_summary = pd.DataFrame(index=btc_metric_table.index)
    btc_component_trust_summary["Accuracy_RMSE"] = btc_metric_table["RMSE"]
    btc_component_trust_summary["Regime_Mean_RMSE"] = btc_robustness_summary["RMSE"].groupby("Model").mean()
    btc_component_trust_summary["Temporal_RMSE_Std"] = btc_temporal_stability_summary["RMSE"].groupby("Model").std()
    btc_component_trust_summary["80% Coverage"] = btc_uncertainty_summary["80% Coverage"]
    btc_component_trust_summary["Transparency"] = btc_explainability_reproducibility["Transparency"]
    btc_component_trust_summary["Reproducibility"] = btc_explainability_reproducibility["Deterministic_or_Fixed_Seed"]
    btc_component_trust_summary

## 11. Statistical and practical significance

**Source:** `09_Statistical_Significance_Test.ipynb`

In [ ]:
def btc_dm_test(actual, forecast_1, forecast_2, power=2):
    aligned=pd.concat([pd.Series(actual).astype(float).rename("actual"),pd.Series(forecast_1).astype(float).rename("forecast_1"),pd.Series(forecast_2).astype(float).rename("forecast_2")],axis=1).dropna()
    if aligned.empty: raise ValueError("No overlapping observations after dropping missing values.")
    loss_1=np.abs(aligned["actual"]-aligned["forecast_1"])**power
    loss_2=np.abs(aligned["actual"]-aligned["forecast_2"])**power
    differential=loss_1-loss_2; n=len(differential)
    if n<2: raise ValueError("At least two observations are required for the Diebold-Mariano test.")
    mean_differential=differential.mean(); variance=differential.var(ddof=1)
    if np.isclose(variance,0):
        statistic=np.inf if mean_differential>0 else -np.inf if mean_differential<0 else 0.0
        p_value=0.0 if not np.isclose(mean_differential,0) else 1.0
    else:
        statistic=mean_differential/np.sqrt(variance/n)
        p_value=2*stats.t.sf(np.abs(statistic),df=n-1)
    return {"DM Statistic":statistic,"p-value":p_value,"Mean Loss Differential":mean_differential,"Sample Size":n}

def btc_cohens_d_absolute_errors(actual, forecast_1, forecast_2):
    difference=np.abs(np.asarray(actual)-np.asarray(forecast_1))-np.abs(np.asarray(actual)-np.asarray(forecast_2))
    return difference.mean()/difference.std(ddof=1) if difference.std(ddof=1)>0 else 0.0

In [ ]:
if RUN_SIGNIFICANCE_TESTS:
    btc_pairs=[("Naive","Persistence-Enhanced LSTM"),("Naive","TimesFM"),("Naive","Chronos-Bolt-Tiny"),("Persistence-Enhanced LSTM","TimesFM"),("Persistence-Enhanced LSTM","Chronos-Bolt-Tiny"),("TimesFM","Chronos-Bolt-Tiny")]
    btc_significance_rows=[]
    for first,second in btc_pairs:
        dm=btc_dm_test(btc_forecast_table["actual"],btc_forecast_table[first],btc_forecast_table[second],power=2)
        rmse_first=btc_metric_table.loc[first,"RMSE"]; rmse_second=btc_metric_table.loc[second,"RMSE"]
        btc_significance_rows.append({"Comparison":f"{first} vs {second}",**dm,"Cohen d absolute errors":btc_cohens_d_absolute_errors(btc_forecast_table["actual"],btc_forecast_table[first],btc_forecast_table[second]),"RMSE difference":rmse_first-rmse_second,"Practical winner":first if rmse_first<rmse_second else second})
    btc_significance_summary=pd.DataFrame(btc_significance_rows).set_index("Comparison")
    btc_significance_summary

## 12. Final Bitcoin decision

The lowest error is not automatically the globally best research choice. Accuracy, robustness, stability, uncertainty, transparency, reproducibility, and statistical evidence are considered together.

In [ ]:
btc_final_decision = {
    "Accuracy leader": btc_metric_table.index[0],
    "Persistence benchmark": "Naive remains the essential reference and strongest accuracy result in the frozen comparison.",
    "Robustness": "Use regime-level evidence; no aggregate score replaces the component table.",
    "Temporal stability": "Compare early, middle, and late test segments before interpreting aggregate rank.",
    "Uncertainty": "Chronos is closer to nominal 80% coverage; TimesFM intervals under-cover; PE-LSTM interval evidence is unavailable.",
    "Statistical evidence": "DM tests and paired effect sizes are reported for the six authoritative pairs.",
    "Main limitation": "One Bitcoin series and a persistence-dominated rolling one-step protocol limit generalisation.",
    "Recommended next action": "Evaluate the same frozen models over additional assets, origins, and explicitly separated multi-step horizons without tuning on the final test.",
}
pd.Series(btc_final_decision)

## 13. Cross-domain-ready Bitcoin outputs

**Provenance:** `18_Cross_Domain_Comparison.ipynb` remains the separate synthesis layer. The following in-memory tables are its clean Bitcoin inputs.

In [ ]:
btc_cross_domain_outputs = {
    "metrics": btc_metric_table.copy(),
    "ranking": btc_final_ranking.copy(),
    "robustness": btc_robustness_summary.copy(),
    "temporal_stability": btc_temporal_stability_summary.copy(),
    "uncertainty": btc_uncertainty_summary.copy(),
    "significance": btc_significance_summary.copy(),
    "trustworthiness_components": btc_component_trust_summary.copy(),
}
{name: table.shape for name,table in btc_cross_domain_outputs.items()}

## Historical and exploratory provenance

- Original raw-price LSTM: exploratory; failed to outperform persistence (`03_Deep_Learning_LSTM.ipynb`).
- Older improved LSTM: superseded by the fixed-seed PE-LSTM (`03b_LSTM_Improved.ipynb`).
- Transformer: original collapse and corrected investigation excluded from the authoritative comparison (`04_Transformers.ipynb`).
- Moving average, SES, Holt, ARIMA, and SARIMA: historical/protocol-limited context (`02_Classical_Models.ipynb`, `05_Advanced_Forecasting_Models.ipynb`).
- Prophet, NeuralForecast, PatchTST, iTransformer, Informer, Autoformer, Moirai/Uni2TS: compatibility, availability, or protocol-limited experiments; no executable scaffolding retained.

These notebooks remain unchanged as the complete development history.